In [0]:
%run ./config

In [0]:
from pyspark.sql.functions import *
def process_landing_to_bronze(table_name):
    cfg = METADATA_CONFIG[table_name]
    source_path = f"{base_path}{cfg['landing_path']}"
    bronze_path = f"{base_path}{cfg['bronze_path']}"
    pipeline_run_id = "Manual_Run"

    try:
        if cfg["file_format"] == "json":
            df_raw = spark.read.format("json").option("multiLine", "true").options(**storage_options).load(source_path)
        elif cfg["file_format"] == "csv":
            df_raw = (spark.read.format(cfg["file_format"])
                      .option("header", "true")
                      .option("inferSchema", "false")
                      .options(**storage_options)
                      .load(source_path))
        else:
            df_raw = (spark.read.format(cfg["file_format"])
                      .options(**storage_options)
                      .load(source_path))

        df_bronze = (df_raw.withColumn("_AdfPipelineRunId",lit(pipeline_run_id))
                     .withColumn("_IngestionTimestamp",current_timestamp())
                     )
        record_count = df_bronze.count()
        df_bronze.write.format("delta").mode("overwrite").options(**storage_options).save(bronze_path)        
        
        print(f"SUCCESS : {table_name} loaded to Bronze")
        print(f"Records Loaded : {record_count}")

    except Exception as e:
        print(f"FAILED : {table_name}")
        print(str(e))

In [0]:
process_landing_to_bronze("orders")

OrderID,CustomerID,ProductID,OrderDate,Quantity,UnitPrice,StoreCode,CurrencyCode,_AdfPipelineRunId,_IngestionTimestamp
3610,99600,2268,2026-05-04,13,526.27,ST029,PHP,Manual_Run,2026-06-27T03:05:07.690Z
3311,2805,1512,2025-01-28,46,1117.67,ST029,TRY,Manual_Run,2026-06-27T03:05:07.690Z
3353,1949,2177,2025-01-31,23,41.1,ST004,KES,Manual_Run,2026-06-27T03:05:07.690Z
2920,2994,2591,2025-10-27,34,2679.71,ST015,EGP,Manual_Run,2026-06-27T03:05:07.690Z
3555,1266,1824,2025-05-31,44,-235.46,ST035,TRY,Manual_Run,2026-06-27T03:05:07.690Z
3246,2173,1215,2025-02-21,30,1465.16,ST017,THB,Manual_Run,2026-06-27T03:05:07.690Z
2154,2039,2551,2025-10-15,3,721.39,ST011,CNY,Manual_Run,2026-06-27T03:05:07.690Z
3749,1973,1648,2027-05-02,50,1448.11,ST027,SAR,Manual_Run,2026-06-27T03:05:07.690Z
2379,2059,2192,2025-02-23,47,2651.47,ST022,EUR,Manual_Run,2026-06-27T03:05:07.690Z
3249,2479,2176,2025-09-21,45,2410.84,ST030,RUB,Manual_Run,2026-06-27T03:05:07.690Z


SUCCESS : orders loaded to Bronze
Records Loaded : 2000


In [0]:
process_landing_to_bronze("products")

ProductID,ProductName,Category,SubCategory,Brand,CostPrice,_AdfPipelineRunId,_IngestionTimestamp
1900,Puma Modern Gym 1900,Sports,Gym,Sony,1606.01,Manual_Run,2026-06-27T03:05:19.416Z
null,LG Cup Accessories 2854,Clothing,Accessories,Samsung,1384.56,Manual_Run,2026-06-27T03:05:19.416Z
2921,Reebok Standard Children 2921,Books,Children,Sony,0,Manual_Run,2026-06-27T03:05:19.416Z
1686,Bosch Project Cameras 1686,Electronics,Cameras,Nike,28.13,Manual_Run,2026-06-27T03:05:19.416Z
2692,HP If Tablets 2692,Electronics,Tablets,HP,-311.83,Manual_Run,2026-06-27T03:05:19.416Z
1499,Nike Fact Water Sports 1499,Sports,Water Sports,Ikea,1813.75,Manual_Run,2026-06-27T03:05:19.416Z
3092,HP Most Gym 3092,Sports,Gym,Bosch,-160.92,Manual_Run,2026-06-27T03:05:19.416Z
1287,Samsung Whole Tablets 1287,Electronics,Tablets,Panasonic,1408.23,Manual_Run,2026-06-27T03:05:19.416Z
2557,Bosch Respond Non-Fiction 2557,Books,Non-Fiction,Adidas,350.06,Manual_Run,2026-06-27T03:05:19.416Z
2672,Ikea Important Men's Wear 2672,Clothing,Men's Wear,LG,-146.53,Manual_Run,2026-06-27T03:05:19.416Z


SUCCESS : products loaded to Bronze
Records Loaded : 2000


In [0]:
process_landing_to_bronze("customers")

CustomerID,FirstName,LastName,Email,Phone,City,State,LastUpdated,_AdfPipelineRunId,_IngestionTimestamp
1846,Jeffrey,Johnson,josephdiaz@example.net,594-665-8414x63452,Samanthaview,Iowa,2025-02-23 11:27:13,Manual_Run,2026-06-27T03:05:58.372Z
3075,Carolyn,null,jenna10@example.net,(685)477-5406,Juliaport,Maine,2025-02-05 15:30:55,Manual_Run,2026-06-27T03:05:58.372Z
2838,Bradley,Jones,null,(635)889-4087x33032,East Mary,New Hampshire,2024-12-16 04:59:11,Manual_Run,2026-06-27T03:05:58.372Z
2413,Gregory,Edwards,thompsondaniel@example.net,337.658.1478x833,North Oscar,Idaho,2026-01-04 22:48:49,Manual_Run,2026-06-27T03:05:58.372Z
2774,null,Mcdowell,daniel01@example.org,001-472-596-3227x609,Christinaborough,Massachusetts,2025-10-22 07:40:04,Manual_Run,2026-06-27T03:05:58.372Z
1596,Andrea,Wells,mackenziefranklin@example.net,+1-453-448-5259x1630,Jimenezside,California,2025-07-19 12:46:09,Manual_Run,2026-06-27T03:05:58.372Z
null,Sharon,Stewart,megandoyle@example.org,879-585-5935x64863,Mcdowellbury,South Dakota,2025-12-24 17:43:21,Manual_Run,2026-06-27T03:05:58.372Z
2350,Stephanie,Landry,brandon97@example.org,902-871-8272,North Robynburgh,Florida,2024-09-12 10:00:27,Manual_Run,2026-06-27T03:05:58.372Z
1392,Brian,Jones,jacksoncraig@example.org,(976)656-8637,East Brendaburgh,Ohio,2025-09-09 00:50:37,Manual_Run,2026-06-27T03:05:58.372Z
2155,Elizabeth,Taylor,brandontapia@example.org,(548)711-3559x495,New Jessica,Maryland,2024-07-22 01:37:06,Manual_Run,2026-06-27T03:05:58.372Z


SUCCESS : customers loaded to Bronze
Records Loaded : 2000


In [0]:
process_landing_to_bronze("exchange_rates")